# Acrobot-v1 PPO Fine-Tuning

In [1]:
# ── Imports ───────────────────────────────────────────────────────────────────
import gymnasium as gym
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.env_util import make_vec_env
from huggingface_sb3 import load_from_hub

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


## 1 — Load the pretrained model & capture a baseline

In [2]:
model_path = load_from_hub(
    repo_id="sb3/ppo-Acrobot-v1",
    filename="ppo-Acrobot-v1.zip"
)

# Load once without an env just to inspect / run baseline episodes
baseline_model = PPO.load(model_path)

baseline_env = gym.make("Acrobot-v1")
baseline_mean, baseline_std = evaluate_policy(
    baseline_model, baseline_env,
    n_eval_episodes=20, deterministic=True
)
baseline_env.close()

print(f"📊 Baseline (pretrained) → Mean reward: {baseline_mean:.2f} ± {baseline_std:.2f}")

c:\Program Files\Python312\Lib\site-packages\stable_baselines3\common\save_util.py:167: UserWarning: Could not deserialize object lr_schedule. Consider using `custom_objects` argument to replace this object.
Exception: code expected at least 16 arguments, got 15
  warnings.warn(
c:\Program Files\Python312\Lib\site-packages\stable_baselines3\common\save_util.py:167: UserWarning: Could not deserialize object clip_range. Consider using `custom_objects` argument to replace this object.
Exception: code expected at least 16 arguments, got 15
  warnings.warn(
c:\Program Files\Python312\Lib\site-packages\stable_baselines3\common\vec_env\patch_gym.py:95: UserWarning: You loaded a model that was trained using OpenAI Gym. We strongly recommend transitioning to Gymnasium by saving that model again.
  warnings.warn(
c:\Program Files\Python312\Lib\site-packages\stable_baselines3\common\evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in

📊 Baseline (pretrained) → Mean reward: -89.35 ± 35.99


In [3]:
for key in vars(baseline_env.spec).keys():
    if hasattr(baseline_env.spec, key):
        print(key, getattr(baseline_env.spec, key))
print(baseline_env.observation_space)
print(baseline_env.action_space)

id Acrobot-v1
entry_point gymnasium.envs.classic_control.acrobot:AcrobotEnv
reward_threshold -100.0
nondeterministic False
max_episode_steps 500
order_enforce True
disable_env_checker False
kwargs {}
additional_wrappers ()
vector_entry_point None
namespace None
name Acrobot
version 1
Box([ -1.        -1.        -1.        -1.       -12.566371 -28.274334], [ 1.        1.        1.        1.       12.566371 28.274334], (6,), float32)
Discrete(3)


In [ ]:
def print_ppo_params(baseline_model):
    print("="*60)
    print("HYPERPARAMETERS (PPO specific)")
    print("="*60)

    # Rollout
    print("\n[Rollout (Data Collection)]")
    print("n_steps:", baseline_model.n_steps)
    print("batch_size:", baseline_model.batch_size)
    print("n_envs:", baseline_model.n_envs)

    # Optimization
    print("\n[Optimization (Training)]")
    print("n_epochs:", baseline_model.n_epochs)
    print("learning_rate:", baseline_model.learning_rate)
    print("vf_coef:", baseline_model.vf_coef)  #checks the error in the value fun

    # Core RL
    print("\n[Core RL (Policy Gradients)]")
    print("gamma:", baseline_model.gamma)
    print("gae_lambda:", baseline_model.gae_lambda)
    print("ent_coef:", baseline_model.ent_coef)
    print("clip_range:", baseline_model.clip_range)

    print("\nTotal timesteps loaded:", getattr(baseline_model, "_total_timesteps", "N/A"))

print_ppo_params(baseline_model)

HYPERPARAMETERS (PPO specific)

[Rollout (Data Collection)]
n_steps: 256
batch_size: 64
n_envs: 16

[Optimization (Training)]
n_epochs: 4
learning_rate: 0.0003
vf_coef: 0.5

[Core RL (Policy Gradients)]
gamma: 0.99
gae_lambda: 0.94
ent_coef: 0.0
clip_range: FloatSchedule(ConstantSchedule(val=0.2))

Total timesteps loaded: 1000000


In [5]:
def print_weights_summary(baseline_model):
    print("="*80)
    print("WEIGHTS SUMMARY")
    print("="*80)

    for name, param in baseline_model.policy.named_parameters():
        if "weight" in name:
            print(f"{name:<60} | shape={tuple(param.shape)} |")

print_weights_summary(baseline_model)

WEIGHTS SUMMARY
mlp_extractor.policy_net.0.weight                            | shape=(64, 6) |
mlp_extractor.policy_net.2.weight                            | shape=(64, 64) |
mlp_extractor.value_net.0.weight                             | shape=(64, 6) |
mlp_extractor.value_net.2.weight                             | shape=(64, 64) |
action_net.weight                                            | shape=(3, 64) |
value_net.weight                                             | shape=(1, 64) |


## 2 — Inspect hyperparameters

In [6]:
def print_ppo_params(model, label=""):
    print(f"{'='*60}")
    print(f"PPO HYPERPARAMETERS  {label}")
    print(f"{'='*60}")
    print(f"  n_steps      : {model.n_steps}")
    print(f"  batch_size   : {model.batch_size}")
    print(f"  n_epochs     : {model.n_epochs}")
    print(f"  learning_rate: {model.learning_rate}")
    print(f"  gamma        : {model.gamma}")
    print(f"  gae_lambda   : {model.gae_lambda}")
    print(f"  ent_coef     : {model.ent_coef}")
    print(f"  clip_range   : {model.clip_range}")
    print(f"  vf_coef      : {model.vf_coef}")

print_ppo_params(baseline_model, "(pretrained)")

PPO HYPERPARAMETERS  (pretrained)
  n_steps      : 256
  batch_size   : 64
  n_epochs     : 4
  learning_rate: 0.0003
  gamma        : 0.99
  gae_lambda   : 0.94
  ent_coef     : 0.0
  clip_range   : FloatSchedule(ConstantSchedule(val=0.2))
  vf_coef      : 0.5


## 3 — Enhanced Early-Stopping Callback

In [7]:
class EarlyStoppingBestWeights(BaseCallback):

    def __init__(
        self,
        eval_env,
        eval_freq: int = 5_000,
        n_eval_episodes: int = 20,
        patience: int = 10,
        floor_reward: float = -115.0,
        floor_patience: int = 5,
        max_regression: float = 20.0,   
        save_path: str = "./best_acrobot_model",
        verbose: int = 1,
    ):
        super().__init__(verbose)
        self.eval_env = eval_env
        self.eval_freq = eval_freq
        self.n_eval_episodes = n_eval_episodes
        self.patience = patience
        self.floor_reward = floor_reward
        self.floor_patience = floor_patience
        self.max_regression = max_regression
        self.save_path = save_path

        self.best_mean_reward = -np.inf
        self.no_improve_count = 0
        self.floor_count = 0
        self.reward_history = []

    def _on_step(self) -> bool:
        if self.n_calls % self.eval_freq != 0:
            return True

        mean_reward, std_reward = evaluate_policy(
            self.model, self.eval_env,
            n_eval_episodes=self.n_eval_episodes,
            deterministic=True,
        )
        self.reward_history.append(mean_reward)

        if self.verbose:    
            print(
                f"  [Eval @ {self.num_timesteps:>7,}]  "
                f"mean={mean_reward:>8.2f} ± {std_reward:.2f}  "
                f"best={self.best_mean_reward:>8.2f}"
            )

        # ── Save best ────────────────────────────────────────────────────────
        if mean_reward > self.best_mean_reward:
            self.best_mean_reward = mean_reward
            self.no_improve_count = 0
            self.model.save(self.save_path)
            if self.verbose:
                print(f"  ✅ New best! Model saved → {self.save_path}.zip")
        else:
            self.no_improve_count += 1

        # ── Floor guard ──────────────────────────────────────────────────────
        if mean_reward < self.floor_reward:
            self.floor_count += 1
        else:
            self.floor_count = 0

        # ── Stop conditions ──────────────────────────────────────────────────
        if self.no_improve_count >= self.patience:
            print(
                f"🛑 Early stop: no improvement for {self.patience} evaluations "
                f"(best={self.best_mean_reward:.2f})."
            )
            return False

        if self.floor_count >= self.floor_patience:
            print(
                f"🛑 Early stop: reward stayed below floor ({self.floor_reward}) "
                f"for {self.floor_patience} evaluations."
            )
            return False

        if (self.best_mean_reward - mean_reward) > self.max_regression:
            print(
                f"🛑 Early stop: catastrophic regression detected — "
                f"dropped {self.best_mean_reward - mean_reward:.1f} pts below best."
            )
            return False

        return True

## 4 — Fine-tune  

In [8]:
# ── Environments ──────────────────────────────────────────────────────────────
# DummyVecEnv with 4 parallel workers gives more stable gradient estimates.
train_env = make_vec_env("Acrobot-v1", n_envs=4)
eval_env  = gym.make("Acrobot-v1")

# ── Load with overridden hyperparams (optimizer is rebuilt here) ──────────────
model = PPO.load(
    model_path,
    env=train_env,
    device="cpu",
    custom_objects={
    "learning_rate": 1e-5,    # lower LR for fine-tuning — more stable updates
    "ent_coef":      0.0005,  # a bit more exploration
    "clip_range":    0.05,    # original PPO default — wider updates
    "n_steps":       1024,    # larger rollout buffer = better gradient estimates
    "batch_size":    256,
},
)

print_ppo_params(model, "(fine-tune)")

# ── Callback ──────────────────────────────────────────────────────────────────
callback = EarlyStoppingBestWeights(
    eval_env=eval_env,
    eval_freq=5_000,
    n_eval_episodes=20,
    patience=8,
    floor_reward=-115.0,
    floor_patience=5,
    max_regression=10.0,
    save_path="./best_acrobot_model",
    verbose=1,
)

# ── Train ─────────────────────────────────────────────────────────────────────
print("\n🚀 Fine-tuning started (up to 500,000 timesteps) …\n")
model.learn(
    total_timesteps=500_000,
    callback=callback,
    reset_num_timesteps=False,   # keep the timestep counter from pretraining
)

train_env.close()
eval_env.close()
print("\n✅ Training finished.")

PPO HYPERPARAMETERS  (fine-tune)
  n_steps      : 1024
  batch_size   : 256
  n_epochs     : 4
  learning_rate: 1e-05
  gamma        : 0.99
  gae_lambda   : 0.94
  ent_coef     : 0.0005
  clip_range   : FloatSchedule(ConstantSchedule(val=0.05))
  vf_coef      : 0.5

🚀 Fine-tuning started (up to 500,000 timesteps) …



c:\Program Files\Python312\Lib\site-packages\stable_baselines3\common\save_util.py:167: UserWarning: Could not deserialize object lr_schedule. Consider using `custom_objects` argument to replace this object.
Exception: code expected at least 16 arguments, got 15
  warnings.warn(
c:\Program Files\Python312\Lib\site-packages\stable_baselines3\common\vec_env\patch_gym.py:95: UserWarning: You loaded a model that was trained using OpenAI Gym. We strongly recommend transitioning to Gymnasium by saving that model again.
  warnings.warn(


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 79.1     |
|    ep_rew_mean     | -78.1    |
| time/              |          |
|    fps             | 2295     |
|    iterations      | 1        |
|    time_elapsed    | 1        |
|    total_timesteps | 1007616  |
---------------------------------
-------------------------------------------
| rollout/                |               |
|    ep_len_mean          | 82            |
|    ep_rew_mean          | -81           |
| time/                   |               |
|    fps                  | 1931          |
|    iterations           | 2             |
|    time_elapsed         | 4             |
|    total_timesteps      | 1011712       |
| train/                  |               |
|    approx_kl            | 1.2642704e-07 |
|    clip_fraction        | 0             |
|    clip_range           | 0.05          |
|    entropy_loss         | -0.0219       |
|    explained_variance   | 0.169         |


## 5 — Reload best checkpoint & evaluate

In [9]:
# Always reload the *best* saved model — not whatever is in memory after training.
best_model = PPO.load("./best_acrobot_model.zip")

test_env = gym.make("Acrobot-v1")
ft_mean, ft_std = evaluate_policy(
    best_model, test_env,
    n_eval_episodes=20, deterministic=True
)
test_env.close()

print("\n" + "="*50)
print(f"  Baseline (pretrained) : {baseline_mean:>8.2f} ± {baseline_std:.2f}")
print(f"  Fine-tuned (best ckpt): {ft_mean:>8.2f} ± {ft_std:.2f}")
delta = ft_mean - baseline_mean
sign  = "+" if delta >= 0 else ""
print(f"  Delta                 : {sign}{delta:.2f}")
print("="*50)


  Baseline (pretrained) :   -89.35 ± 35.99
  Fine-tuned (best ckpt):   -80.75 ± 23.52
  Delta                 : +8.60


## 6 — Visual episode rollout (sanity check)

In [10]:
render_env = gym.make("Acrobot-v1", render_mode="human")

for ep in range(5):
    obs, _ = render_env.reset()
    total_reward, steps = 0, 0
    while True:
        action, _ = best_model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, _ = render_env.step(int(action))
        total_reward += reward
        steps += 1
        if terminated or truncated:
            break
    print(f"Episode {ep+1:>2} | Reward: {total_reward:>7.1f} | Steps: {steps}")

render_env.close()

Episode  1 | Reward:   -64.0 | Steps: 65
Episode  2 | Reward:   -72.0 | Steps: 73
Episode  3 | Reward:   -97.0 | Steps: 98
Episode  4 | Reward:   -74.0 | Steps: 75
Episode  5 | Reward:  -126.0 | Steps: 127
